# Polaris API → SQL → MinIO Access Map (Phase 1)







**Purpose:** for every Polaris Management API and Iceberg REST Catalog API operation, record

1. the exact SQL issued against the PostgreSQL metastore, and which tables it touches,
2. the MinIO objects/paths it accesses,
3. whether any statement lacks a selective index — the headline question.

**LOCAL ONLY.** This notebook enables debug logging, seeds ~16,000 entities and tears them down.
It must never run against the shared company DEV or PROD realms.

**Run order:** every cell top-to-bottom (`Restart & Run All`). Section 2 (schema drift) runs
*before* anything else, because a missing index caused by an unmigrated local schema is a
different problem from one upstream never created.

**Before running:** complete `doc-instrumentation-runbook.md`. The preflight in section 1 fails
loudly if the log streams are not live — a silent miss would look like "this API issued no SQL",
which is exactly the wrong conclusion.

## 0. Bootstrap

In [1]:
import gc
import json
import pathlib
import sys
from datetime import datetime

sys.path.insert(0, str(pathlib.Path.cwd().parent.parent / "src"))

from polaris_test_utils import *  # noqa: F403 — init_env, root_token, PG_* globals

init_env("local")                     # default env; never "dev"/"prod" here
require_not_prod("api-sql-profile: enables debug logging and seeds ~16k entities")

from api_trace import (
    FileStream,
    Tracer,
    api_minio_matrix,
    api_table_matrix,
    records_to_rows,
    statement_inventory,
    unknown_tables,
)
from iceberg_rest import (IcebergREST, build_create_table_payload,
                          build_schema, build_scan_report)
from polaris_rest import PolarisREST
from polaris_seed import SeedSpec, find_strays, seed, teardown, verify_counts
from schema_audit import (
    analyze_tables,
    audit_statements,
    check_hypotheses,
    compare_schema,
    describe_schema,
    rank_statements,
    statistics_are_stale,
    table_stats,
)

# Second guard, independent of require_not_prod: this notebook is destructive.
assert any(h in POLARIS_URL for h in ("localhost", "127.0.0.1", "192.168.")), (
    f"Refusing to run against {POLARIS_URL} — api-sql-profile is local-only."
)

TOKEN = root_token()
pc = PolarisREST(POLARIS_URL, REALM, token=TOKEN)
ic = IcebergREST(POLARIS_URL, REALM, token=TOKEN)

print(f"Polaris  : {POLARIS_URL}  realm={REALM}")
print(f"Postgres : {PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"Started  : {datetime.now():%Y-%m-%d %H:%M:%S}")

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
Polaris  : http://192.168.139.2:8181  realm=POLARIS
Postgres : 192.168.139.2:5432/polaris
Started  : 2026-08-20 13:51:29


### Capture directory and stream wiring

`CAPTURE_DIR` holds the raw log captures. **It must be gitignored** — captures contain bound
parameters, and statements against `principal_authentication_data` carry secret hashes and salts.
`api_trace` redacts those before they reach any record or document, but the raw files themselves
are not safe to commit.

In [2]:
import os


def _find_capture_dir():
    """Locate the capture directory that actually holds this run's logs.

    Three things have to line up and previously did not:
      * a notebook's cwd is its OWN directory, so a relative "capture" resolves
        to diagnostics/api-sql-profile/capture while the log may sit at the
        repo root;
      * `capture.sh rotate <dir>` writes to a NEW directory (e.g.
        capture-seeded), which no hardcoded "capture" would ever find -- it
        would silently analyse the stale pre-seed logs instead, which is worse
        than failing;
      * an explicit choice must win over both.

    Order: $CAPTURE_DIR, then the most RECENTLY MODIFIED capture* directory
    holding a non-empty polaris.log, in this dir or the repo root.
    """
    env = os.environ.get("CAPTURE_DIR")
    if env:
        return pathlib.Path(env)

    roots = [pathlib.Path("."), pathlib.Path.cwd().parents[1]]
    cands = []
    for root in roots:
        for d in sorted(root.glob("capture*")):
            log = d / "polaris.log"
            if d.is_dir() and log.exists() and log.stat().st_size > 0:
                cands.append((log.stat().st_mtime, d))
    if cands:
        cands.sort(reverse=True)
        chosen = cands[0][1]
        if len(cands) > 1:
            others = ", ".join(str(d) for _, d in cands[1:])
            print(f"note: several capture dirs have logs; using the newest "
                  f"({chosen}). Others: {others}")
        return chosen
    return pathlib.Path("capture")


CAPTURE_DIR = _find_capture_dir()
CAPTURE_DIR.mkdir(exist_ok=True)
print(f"capture dir: {CAPTURE_DIR.resolve()}")

POLARIS_LOG = CAPTURE_DIR / "polaris.log"
# capture.sh writes one file per PostgreSQL replica (pg-0.log, pg-1.log, ...)
# because Pgpool load-balances reads, so a statement can land on any node and a
# shared append target both interleaves lines and loses which node ran it.
# Older runs produced a single pg.log; accept either.
PG_LOGS = sorted(CAPTURE_DIR.glob("pg*.log"))
PG_LOG = PG_LOGS[0] if PG_LOGS else CAPTURE_DIR / "pg.log"
MINIO_TRACE = CAPTURE_DIR / "minio.json"


class MultiStream:
    """FileStream over SEVERAL files, concatenating each one's new output.

    Pgpool load-balances reads, so a statement issued by Polaris can execute on
    any of the three replicas and its `duration:` line lands in that node's log.
    Reading only one file would attribute timings to a fraction of the
    statements and silently leave the rest blank -- which looks identical to
    "durations are off". Same tiny interface as FileStream: exists / mark /
    read_since_mark.
    """

    def __init__(self, paths):
        self.streams = [FileStream(str(x)) for x in paths]

    def exists(self):
        return any(s.exists() for s in self.streams)

    def mark(self):
        for s in self.streams:
            s.mark()

    def read_since_mark(self):
        return "".join(s.read_since_mark() for s in self.streams)


# Generated reports land here, one timestamped set per run, so a re-run never
# overwrites the evidence from the previous one -- comparing before/after an
# index change depends on both still existing.
REPORTS_DIR = pathlib.Path(os.environ.get("REPORTS_DIR", "reports"))
REPORTS_DIR.mkdir(exist_ok=True)
_rg = REPORTS_DIR / ".gitignore"
if not _rg.exists():
    # Created empty-but-documented on purpose: which reports are worth
    # committing is a judgement call, so the rules are left to you. Never
    # overwritten once it exists.
    _rg.write_text(
        "# Generated reports from 01_api_access_map / 02_index_audit.\n"
        "# Managed by hand -- add rules as you see fit, e.g.:\n"
        "#   *.md\n"
        "#   !doc-*-latest.md\n",
        encoding="utf-8",
    )
RUN_STAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
print(f"reports   : {REPORTS_DIR.resolve()}  (stamp {RUN_STAMP})")

PG_STREAM = MultiStream(PG_LOGS) if PG_LOGS else None
print(f"pg logs   : {[x.name for x in PG_LOGS] or 'none'}")

# Direct-to-Postgres for the mapping run (decision #6): bypassing PgBouncer removes
# the session-attribution ambiguity. The pooler's own cost is measured in a later phase.
import psycopg2

conn = psycopg2.connect(
    host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASSWORD
)
conn.autocommit = True

gitignore = CAPTURE_DIR / ".gitignore"
if not gitignore.exists():
    # `!.gitignore` so the rule itself stays committable -- a bare `*`
    # ignores this file too, and the rule never reaches a fresh clone.
    gitignore.write_text("*\n!.gitignore\n", encoding="utf-8")
print(f"capture dir: {CAPTURE_DIR.resolve()} (gitignored)")

capture dir: /Users/kade/hynix/polaris-practice/polaris-learning/diagnostics/api-sql-profile/capture-seeded
reports   : /Users/kade/hynix/polaris-practice/polaris-learning/diagnostics/api-sql-profile/reports  (stamp 20260820-135131)
pg logs   : ['pg-0.log', 'pg-1.log', 'pg-2.log']
capture dir: /Users/kade/hynix/polaris-practice/polaris-learning/diagnostics/api-sql-profile/capture-seeded (gitignored)


## 1. Instrumentation preflight

In [3]:
def preflight():
    """Verify each stream is actually producing output before the sweep starts.

    A stream that is configured but not running yields empty captures, which read
    as "this API touched no tables" — the most misleading possible failure. Fail
    here instead.
    """
    issues, warnings = [], []

    if not POLARIS_LOG.exists() or POLARIS_LOG.stat().st_size == 0:
        issues.append(
            f"{POLARIS_LOG} missing/empty — is the log tail running?\n"
            f"       searched: ./capture and {pathlib.Path.cwd().parents[1] / 'capture'}\n"
            "       start it with:  ./capture.sh start   (or rotate <dir>)"
        )
    else:
        text = POLARIS_LOG.read_text(errors="replace")[-200_000:]
        if "DatasourceOperations" not in text:
            issues.append(
                "Polaris log has no DatasourceOperations lines. SQL DEBUG logging is "
                "off. Set QUARKUS_LOG_CATEGORY__ORG_APACHE_POLARIS_PERSISTENCE_"
                "RELATIONAL_JDBC__LEVEL=DEBUG and confirm quarkus.log.min-level "
                "allows DEBUG (build-time property — a runtime category setting is "
                "silently ignored if min-level is higher)."
            )

    if not PG_LOGS or not any(x.stat().st_size for x in PG_LOGS):
        warnings.append(
            f"no non-empty pg*.log in {CAPTURE_DIR} — proceeding WITHOUT "
            "server-side durations (the `ms` column stays empty). Enable with: "
            "PGDUR=1 ./capture.sh pgon  &&  ./capture.sh rotate. "
            "Table mapping still works; timing columns will be empty."
        )
    if not MINIO_TRACE.exists() or MINIO_TRACE.stat().st_size == 0:
        warnings.append(
            f"{MINIO_TRACE} missing/empty — proceeding WITHOUT MinIO attribution. "
            "The 'which storage path' question cannot be answered from this run."
        )

    for w in warnings:
        print(f"WARN  {w}")
    if issues:
        raise RuntimeError("Preflight failed:\n  - " + "\n  - ".join(issues))
    print("Preflight OK")


preflight()

tracer = Tracer(
    polaris_log=FileStream(str(POLARIS_LOG)),
    pg_log=PG_STREAM if (PG_STREAM and PG_STREAM.exists()) else None,
    minio_trace=FileStream(str(MINIO_TRACE)) if MINIO_TRACE.exists() else None,
    settle_s=0.30,   # logs flush asynchronously; without this the tail of a call is missed
)

Preflight OK


## 2. Schema drift and index presence

**This runs first, deliberately.** A missing index has two very different causes:

* the running Polaris version's schema never defined it → an upstream gap
* this instance was created under an older schema and never migrated → a local migration gap

Only the drift check distinguishes them. Drawing a conclusion from an `EXPLAIN` before running
this risks reporting a local migration problem as an upstream bug.

In [4]:
snapshot = describe_schema(conn)
drift = compare_schema(snapshot)

print(f"schema version : {drift['version_found']} (expected {drift['version_expected']})")
print(f"verdict        : {drift['verdict']}")
print(f"tables         : {snapshot['tables']}")
print(f"indexes        : {len(snapshot['indexes'])}")
for note in drift["notes"]:
    print(f"  note: {note}")

if drift["missing_indexes"]:
    print("\nMISSING EXPECTED ACCESS PATHS:")
    for i in drift["missing_indexes"]:
        print(f"  - {i['name']}  {i['table']}({', '.join(i['columns'])})  [{i['kind']}]")

if drift["extra_indexes"]:
    print("\nIndexes present but not in the expected v2 set (fine — often hand-added):")
    for i in drift["extra_indexes"]:
        print(f"  - {i['name']}  {i['table']}({', '.join(i['columns'])})")

schema version : 3 (expected 3)
verdict        : OK
tables         : ['entities', 'events', 'grant_records', 'policy_mapping_record', 'principal_authentication_data', 'version']
indexes        : 10
  note: ['events'] present — created by the persistence event listener (eventListener.type: persistence-in-memory-buffer), which is configuration, not schema drift. Note its writes are flushed on a timer, so they are attributed separately during tracing.

Indexes present but not in the expected v2 set (fine — often hand-added):
  - events_pkey  events(event_id)


In [5]:
import pandas as pd

df_idx = pd.DataFrame(snapshot["indexes"])[["table", "name", "columns", "is_unique", "is_primary"]]
df_idx["columns"] = df_idx["columns"].apply(", ".join)
df_idx.sort_values(["table", "name"]).reset_index(drop=True)

,table,name,columns,is_unique,is_primary
0,entities,constraint_name,"realm_id, catalog_id, parent_id, type_code, name",True,False
1,entities,entities_pkey,"realm_id, id",True,True
2,entities,idx_entities,"realm_id, catalog_id, id",False,False
3,entities,idx_locations,"realm_id, parent_id, location_without_scheme",False,False
4,events,events_pkey,event_id,True,True
5,grant_records,grant_records_pkey,"realm_id, securable_catalog_id, securable_id, ...",True,True
6,policy_mapping_record,idx_policy_mapping_record,"realm_id, policy_type_code, policy_catalog_id,...",False,False
7,policy_mapping_record,policy_mapping_record_pkey,"realm_id, target_catalog_id, target_id, policy...",True,True
8,principal_authentication_data,principal_authentication_data_pkey,"realm_id, principal_client_id",True,True
9,version,version_pkey,version_key,True,True


## 3. Seed fixture — OPTIONAL, SLOW, DESTRUCTIVE

An `EXPLAIN` against a near-empty metastore proves nothing: PostgreSQL picks a sequential scan on
a small table because that is genuinely cheaper. The fixture exists so the planner's choice
carries information.

**Cost:** ~17,000–33,000 API calls (15–30 min), ~10,000 MinIO objects, ~16,000 `entities` rows and
~25,000 `grant_records` rows left in the realm until teardown (section 9).

Set `RUN_SEED = True` to build it. Leave it `False` to map the API surface against whatever data
already exists — the mapping is valid either way; only the *index verdicts* need the volume.

`SEED_TABLES = False` gives a metadata-only seed: ~10x faster, still fully populates
`grant_records`, which is the table the grantee-index hypothesis is about.

In [6]:
RUN_SEED = False          # <-- set True to build the fixture
SEED_TABLES = True        # <-- False for a much faster metadata-only seed
LEDGER = str(CAPTURE_DIR / "seed_ledger.json")

spec = SeedSpec(n_users=1000, namespaces_per_catalog=2, tables_per_namespace=5,
                create_tables=SEED_TABLES)
print("projected:", json.dumps(spec.expected_counts(), indent=2))

if RUN_SEED:
    def on_progress(done, total, res):
        rate = res.calls / max(res.elapsed_s, 0.001)
        print(f"  {done}/{total} users  calls={res.calls}  "
              f"{rate:.0f} calls/s  elapsed={res.elapsed_s:.0f}s")

    result = seed(pc, ic, spec, ledger_path=LEDGER, bucket=BUCKET,
                  minio_endpoint=MINIO_ENDPOINT, on_progress=on_progress)
    print("\n" + result.summary())
    if result.invalid_privileges:
        print(
            "NOTE: the server rejected these privilege names — grant_records volume "
            f"is lower than projected: {sorted(set(result.invalid_privileges))}"
        )
    if result.failed_users:
        print(f"FAILED users (re-run this cell to retry): {result.failed_users[:5]}")
else:
    print("Seeding skipped. Index verdicts may come back TOO_SMALL.")

projected: {
  "principals": 1000,
  "principal_roles": 1000,
  "catalogs": 1000,
  "catalog_roles": 1000,
  "namespaces": 2000,
  "tables": 10000,
  "entities_total": 16000,
  "grant_records": 25000
}
Seeding skipped. Index verdicts may come back TOO_SMALL.


In [7]:
verification = verify_counts(conn, spec) if RUN_SEED else None
if verification:
    print(json.dumps(verification["actual"], indent=2))
    for note in verification["notes"]:
        print(f"  note: {note}")
else:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM polaris_schema.entities")
        n_ent = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM polaris_schema.grant_records")
        n_grant = cur.fetchone()[0]
    print(f"current volume: entities={n_ent}  grant_records={n_grant}")
    if n_grant < 5000:
        print(
            "  note: grant_records is small — the grantee access-path hypothesis "
            "cannot be settled at this volume (verdicts will read TOO_SMALL)."
        )

current volume: entities=7269  grant_records=30009


## 4. Probe fixture

A small dedicated catalog for the sweep itself, separate from the bulk seed. Each traced operation
needs a known target, and mixing probe entities into the seeded set would make teardown ambiguous.

In [8]:
PROBE = f"apiprofile{int(datetime.now().timestamp())}"
PROBE_CAT, PROBE_NS, PROBE_NS2 = f"{PROBE}_cat", "probe_ns", "probe_ns2"
PROBE_TBL, PROBE_VIEW = "probe_tbl", "probe_view"
PROBE_PRINCIPAL, PROBE_PROLE, PROBE_CROLE = f"{PROBE}_p", f"{PROBE}_pr", f"{PROBE}_cr"

schema = build_schema([(1, "id", "long", True), (2, "val", "string", False)])

# Reuse the seeder's lag handling rather than writing a third variant of it.
# _attempt already encodes what a thousand-user seed taught us: on 5xx / 403 /
# 404 the prerequisite may simply not have replicated yet, so verify existence
# and retry with backoff; a duplicate-key body means the write already landed;
# and any other 4xx is a real error that must not be retried.
from polaris_seed import SeedResult, _attempt, _ensure_catalog

_probe_result = SeedResult()


def ck(call, what, exists=None):
    """Run a fixture call, tolerating replication lag, and fail loudly if real.

    `call` is a CALLABLE, not a response -- it has to be re-issuable. The old
    version took an already-made response and asserted on it, so a 500 that had
    actually committed (or would have succeeded a moment later) aborted the
    whole notebook. That is what this just hit:

        create namespace probe_ns -> [500] NullPointerException:
        getPassthroughResolvedPath(...) is null

    which is the catalog not yet being resolvable on the replica serving the
    read -- the namespace often exists regardless.
    """
    ok, r = _attempt(call, what, _probe_result, exists=exists)
    if not ok:
        raise AssertionError(f"{what} -> [{r.status_code}] {r.text[:300]}")
    return r


# Not a bare create: a catalog whose entity commits while its grant bootstrap
# does not is permanently unusable, and _ensure_catalog verifies catalog_admin
# is really there (deleting and recreating if not) instead of trusting a 200
# from get_catalog.
_ensure_catalog(pc, PROBE_CAT, BUCKET, MINIO_ENDPOINT, _probe_result)

# Creating a catalog does NOT grant rights inside it. root holds service_admin,
# which covers CATALOG_CREATE(25)/CATALOG_LIST(27) at the ROOT container, but
# namespace and table operations are CATALOG-scoped and authorize against
# grant_records.
ck(lambda: pc.grant_privilege(PROBE_CAT, "catalog_admin", "CATALOG_MANAGE_CONTENT"),
   "grant CATALOG_MANAGE_CONTENT")

for ns in (PROBE_NS, PROBE_NS2):
    ck(lambda ns=ns: ic.create_namespace(PROBE_CAT, ns),
       f"create namespace {ns}",
       lambda ns=ns: ic.namespace_exists(PROBE_CAT, ns))

ck(lambda: ic.create_table(PROBE_CAT, PROBE_NS,
                           build_create_table_payload(PROBE_TBL, schema)),
   f"create table {PROBE_TBL}",
   lambda: ic.table_exists(PROBE_CAT, PROBE_NS, PROBE_TBL))

if _probe_result.lag_recovered:
    print(f"  (absorbed {len(_probe_result.lag_recovered)} lag failures while "
          f"building the fixture: "
          f"{sorted({e['status'] for e in _probe_result.lag_recovered})})")
if _probe_result.repaired_catalogs:
    print(f"  (repaired half-created catalog: {_probe_result.repaired_catalogs})")
print(f"probe fixture ready: {PROBE_CAT}")


  (absorbed 2 lag failures while building the fixture: [500])
probe fixture ready: apiprofile1787201504_cat


## 5. The sweep

Full sweep of both API surfaces (decision #1), with the core CRUD subset marked `core=True` so the
reports can rank it first.

Each operation is one cold call inside a trace window. **Phase 1 measures mapping, not latency** —
the repeat/cache protocol is a later phase (decisions #3 and #7), so a single call per operation is
the right granularity here.

In [9]:
def probe(api, method, path, fn, core=False):
    """Trace one API call. Failures are recorded and the sweep continues —
    a 403 or 404 still tells you which tables the authorization path touched."""
    try:
        with tracer.trace(api, method, path) as rec:
            resp = fn()
            rec.status = getattr(resp, "status_code", None)
    except Exception as exc:
        print(f"  ! {api}: {type(exc).__name__}: {exc}")
        return None
    rec.core = core
    flag = "*" if core else " "
    print(f"{flag} {api:<44} {rec.status}  sql={rec.sql_count:<3} "
          f"minio={rec.minio_count:<3} {rec.wall_ms:.0f}ms  {rec.cache_verdict}")
    return rec


print("core ops marked with *\n")
print("--- Iceberg REST: config & namespaces ---")
probe("iceberg.get_config", "GET", "/v1/config",
      lambda: ic.get_config(warehouse=PROBE_CAT), core=True)
probe("iceberg.list_namespaces", "GET", "/v1/{cat}/namespaces",
      lambda: ic.list_namespaces(PROBE_CAT), core=True)
probe("iceberg.load_namespace", "GET", "/v1/{cat}/namespaces/{ns}",
      lambda: ic.load_namespace(PROBE_CAT, PROBE_NS), core=True)
probe("iceberg.head_namespace", "HEAD", "/v1/{cat}/namespaces/{ns}",
      lambda: ic.head_namespace(PROBE_CAT, PROBE_NS))
probe("iceberg.update_namespace_properties", "POST", "/v1/{cat}/namespaces/{ns}/properties",
      lambda: ic.update_namespace_properties(PROBE_CAT, PROBE_NS, updates={"k": "v"}))
probe("iceberg.create_namespace", "POST", "/v1/{cat}/namespaces",
      lambda: ic.create_namespace(PROBE_CAT, "probe_ns_tmp"), core=True)
probe("iceberg.drop_namespace", "DELETE", "/v1/{cat}/namespaces/{ns}",
      lambda: ic.drop_namespace(PROBE_CAT, "probe_ns_tmp"), core=True)

core ops marked with *

--- Iceberg REST: config & namespaces ---
* iceberg.get_config                           200  sql=14  minio=0   28ms  MISS
* iceberg.list_namespaces                      200  sql=26  minio=0   32ms  MISS
* iceberg.load_namespace                       200  sql=16  minio=0   12ms  MISS
  iceberg.head_namespace                       204  sql=19  minio=0   12ms  MISS
  iceberg.update_namespace_properties          200  sql=20  minio=0   33ms  MISS
* iceberg.create_namespace                     500  sql=30  minio=0   54ms  MISS
* iceberg.drop_namespace                       204  sql=33  minio=0   50ms  MISS


TraceRecord(api='iceberg.drop_namespace', method='DELETE', path='/v1/{cat}/namespaces/{ns}', status=204, t0=1787201516.065879, t1=1787201516.1163032, error=None, sql=[SqlStatement(seq=0, sql='SELECT id, catalog_id, parent_id, type_code, name, entity_version, sub_type_code, create_timestamp, drop_timestamp, purge_timestamp, to_purge_timestamp, last_update_timestamp, properties, internal_properties, grant_records_version, location_without_scheme FROM POLARIS_SCHEMA.ENTITIES WHERE realm_id = ? AND name = ? AND catalog_id = ? AND parent_id = ? AND type_code = ?', params='POLARIS, root, 0, 0, 2', table='entities', verb='SELECT', duration_ms=0.151, source='polaris', request_id='04e6a399-b545-4cf4-80c2-1464ed19c42c_0000000000000068126', trace_id=None), SqlStatement(seq=1, sql='SELECT id, catalog_id, parent_id, type_code, name, entity_version, sub_type_code, create_timestamp, drop_timestamp, purge_timestamp, to_purge_timestamp, last_update_timestamp, properties, internal_properties, grant_reco

In [10]:
print("--- Iceberg REST: tables ---")
probe("iceberg.list_tables", "GET", "/v1/{cat}/namespaces/{ns}/tables",
      lambda: ic.list_tables(PROBE_CAT, PROBE_NS), core=True)
probe("iceberg.load_table", "GET", "/v1/{cat}/namespaces/{ns}/tables/{tbl}",
      lambda: ic.load_table(PROBE_CAT, PROBE_NS, PROBE_TBL), core=True)
probe("iceberg.load_table[snapshots=refs]", "GET", "/v1/{cat}/.../tables/{tbl}?snapshots=refs",
      lambda: ic.load_table(PROBE_CAT, PROBE_NS, PROBE_TBL, snapshots="refs"))
probe("iceberg.head_table", "HEAD", "/v1/{cat}/namespaces/{ns}/tables/{tbl}",
      lambda: ic.head_table(PROBE_CAT, PROBE_NS, PROBE_TBL))
probe("iceberg.create_table", "POST", "/v1/{cat}/namespaces/{ns}/tables",
      lambda: ic.create_table(PROBE_CAT, PROBE_NS,
                              build_create_table_payload("probe_tbl2", schema)), core=True)
probe("iceberg.stage_create_table", "POST", "/v1/{cat}/namespaces/{ns}/tables[stage]",
      lambda: ic.stage_create_table(PROBE_CAT, PROBE_NS,
                                    build_create_table_payload("probe_staged", schema)))
probe("iceberg.commit_table", "POST", "/v1/{cat}/namespaces/{ns}/tables/{tbl}",
      lambda: ic.commit_table(PROBE_CAT, PROBE_NS, PROBE_TBL,
                              [{"action": "set-properties", "updates": {"p": "1"}}]), core=True)
probe("iceberg.rename_table", "POST", "/v1/{cat}/tables/rename",
      lambda: ic.rename_table(PROBE_CAT, PROBE_NS, "probe_tbl2", PROBE_NS, "probe_tbl3"))
# A ScanReport needs all SEVEN required fields; the two-field stub used before
# was a 400 (and the field is `projected-field-names`, not `projection`).
# snapshot-id must be a real snapshot: the probe table has been committed to by
# this point, so read it back rather than inventing an id.
_probe_meta = ic.load_table(PROBE_CAT, PROBE_NS, PROBE_TBL).json().get("metadata", {})
_probe_snap = _probe_meta.get("current-snapshot-id", -1)
probe("iceberg.report_metrics", "POST", "/v1/{cat}/.../tables/{tbl}/metrics",
      lambda: ic.report_metrics(
          PROBE_CAT, PROBE_NS, PROBE_TBL,
          build_scan_report(PROBE_TBL, _probe_snap)))
probe("iceberg.load_table[missing]", "GET", "/v1/{cat}/.../tables/{missing}",
      lambda: ic.load_table(PROBE_CAT, PROBE_NS, "does_not_exist"))
probe("iceberg.drop_table", "DELETE", "/v1/{cat}/namespaces/{ns}/tables/{tbl}",
      lambda: ic.drop_table(PROBE_CAT, PROBE_NS, "probe_tbl3"), core=True)

--- Iceberg REST: tables ---
* iceberg.list_tables                          200  sql=18  minio=0   12ms  MISS
* iceberg.load_table                           200  sql=29  minio=1   94ms  MISS
  iceberg.load_table[snapshots=refs]           200  sql=16  minio=1   26ms  MISS
  iceberg.head_table                           204  sql=16  minio=1   42ms  MISS
* iceberg.create_table                         200  sql=46  minio=1   71ms  MISS
  iceberg.stage_create_table                   200  sql=38  minio=0   52ms  MISS
* iceberg.commit_table                         200  sql=30  minio=3   78ms  MISS
  iceberg.rename_table                         200  sql=26  minio=0   31ms  MISS
  iceberg.report_metrics                       204  sql=12  minio=0   9ms  MISS
  iceberg.load_table[missing]                  404  sql=16  minio=0   32ms  MISS
* iceberg.drop_table                           204  sql=87  minio=0   60ms  MISS


TraceRecord(api='iceberg.drop_table', method='DELETE', path='/v1/{cat}/namespaces/{ns}/tables/{tbl}', status=204, t0=1787201528.443229, t1=1787201528.503027, error=None, sql=[SqlStatement(seq=0, sql='SELECT id, catalog_id, parent_id, type_code, name, entity_version, sub_type_code, create_timestamp, drop_timestamp, purge_timestamp, to_purge_timestamp, last_update_timestamp, properties, internal_properties, grant_records_version, location_without_scheme FROM POLARIS_SCHEMA.ENTITIES WHERE realm_id = ? AND name = ? AND catalog_id = ? AND parent_id = ? AND type_code = ?', params='POLARIS, root, 0, 0, 2', table='entities', verb='SELECT', duration_ms=0.173, source='polaris', request_id='04e6a399-b545-4cf4-80c2-1464ed19c42c_0000000000000068138', trace_id=None), SqlStatement(seq=1, sql='SELECT id, catalog_id, parent_id, type_code, name, entity_version, sub_type_code, create_timestamp, drop_timestamp, purge_timestamp, to_purge_timestamp, last_update_timestamp, properties, internal_properties, gr

In [11]:
print("--- Iceberg REST: views ---")
view_payload = {
    "name": PROBE_VIEW,
    "schema": schema,
    "view-version": {
        "version-id": 1, "schema-id": 0, "timestamp-ms": 0,
        "summary": {"operation": "create"},
        "representations": [{"type": "sql", "sql": "SELECT 1", "dialect": "spark"}],
        "default-namespace": [PROBE_NS],
    },
    "properties": {},
}
probe("iceberg.create_view", "POST", "/v1/{cat}/namespaces/{ns}/views",
      lambda: ic.create_view(PROBE_CAT, PROBE_NS, view_payload))
probe("iceberg.list_views", "GET", "/v1/{cat}/namespaces/{ns}/views",
      lambda: ic.list_views(PROBE_CAT, PROBE_NS))
probe("iceberg.load_view", "GET", "/v1/{cat}/namespaces/{ns}/views/{view}",
      lambda: ic.load_view(PROBE_CAT, PROBE_NS, PROBE_VIEW))
probe("iceberg.head_view", "HEAD", "/v1/{cat}/namespaces/{ns}/views/{view}",
      lambda: ic.head_view(PROBE_CAT, PROBE_NS, PROBE_VIEW))
probe("iceberg.rename_view", "POST", "/v1/{cat}/views/rename",
      lambda: ic.rename_view(PROBE_CAT, PROBE_NS, PROBE_VIEW, PROBE_NS, "probe_view2"))
probe("iceberg.drop_view", "DELETE", "/v1/{cat}/namespaces/{ns}/views/{view}",
      lambda: ic.drop_view(PROBE_CAT, PROBE_NS, "probe_view2"))

--- Iceberg REST: views ---
  iceberg.create_view                          500  sql=42  minio=1   47ms  MISS
  iceberg.list_views                           200  sql=60  minio=0   15ms  MISS
  iceberg.load_view                            200  sql=30  minio=2   37ms  MISS
  iceberg.head_view                            204  sql=30  minio=2   29ms  MISS
  iceberg.rename_view                          204  sql=22  minio=0   18ms  MISS
  iceberg.drop_view                            204  sql=33  minio=0   46ms  MISS


TraceRecord(api='iceberg.drop_view', method='DELETE', path='/v1/{cat}/namespaces/{ns}/views/{view}', status=204, t0=1787201533.240885, t1=1787201533.286687, error=None, sql=[SqlStatement(seq=0, sql='SELECT id, catalog_id, parent_id, type_code, name, entity_version, sub_type_code, create_timestamp, drop_timestamp, purge_timestamp, to_purge_timestamp, last_update_timestamp, properties, internal_properties, grant_records_version, location_without_scheme FROM POLARIS_SCHEMA.ENTITIES WHERE realm_id = ? AND name = ? AND catalog_id = ? AND parent_id = ? AND type_code = ?', params='POLARIS, root, 0, 0, 2', table='entities', verb='SELECT', duration_ms=0.146, source='polaris', request_id='04e6a399-b545-4cf4-80c2-1464ed19c42c_0000000000000068144', trace_id=None), SqlStatement(seq=1, sql='SELECT id, catalog_id, parent_id, type_code, name, entity_version, sub_type_code, create_timestamp, drop_timestamp, purge_timestamp, to_purge_timestamp, last_update_timestamp, properties, internal_properties, gra

In [12]:
print("--- Management API: catalogs, principals, roles, grants ---")
probe("mgmt.list_catalogs", "GET", "/v1/catalogs", lambda: pc.list_catalogs(), core=True)
probe("mgmt.get_catalog", "GET", "/v1/catalogs/{cat}",
      lambda: pc.get_catalog(PROBE_CAT), core=True)
probe("mgmt.create_principal", "POST", "/v1/principals",
      lambda: pc.create_principal(PROBE_PRINCIPAL), core=True)
probe("mgmt.get_principal", "GET", "/v1/principals/{p}",
      lambda: pc.get_principal(PROBE_PRINCIPAL))
probe("mgmt.list_principals", "GET", "/v1/principals", lambda: pc.list_principals())
probe("mgmt.create_principal_role", "POST", "/v1/principal-roles",
      lambda: pc.create_principal_role(PROBE_PROLE))
probe("mgmt.get_principal_role", "GET", "/v1/principal-roles/{r}",
      lambda: pc.get_principal_role(PROBE_PROLE))
probe("mgmt.list_principal_roles", "GET", "/v1/principal-roles",
      lambda: pc.list_principal_roles())
probe("mgmt.assign_principal_role", "PUT", "/v1/principals/{p}/principal-roles",
      lambda: pc.assign_principal_role_to_principal(PROBE_PRINCIPAL, PROBE_PROLE))
probe("mgmt.create_catalog_role", "POST", "/v1/catalogs/{cat}/catalog-roles",
      lambda: pc.create_catalog_role(PROBE_CAT, PROBE_CROLE))
probe("mgmt.list_catalog_roles", "GET", "/v1/catalogs/{cat}/catalog-roles",
      lambda: pc.list_catalog_roles(PROBE_CAT))
probe("mgmt.assign_catalog_role", "PUT", "/v1/principal-roles/{r}/catalog-roles/{cat}",
      lambda: pc.assign_catalog_role_to_principal_role(PROBE_CAT, PROBE_PROLE, PROBE_CROLE))
probe("mgmt.grant_privilege", "PUT", "/v1/catalogs/{cat}/catalog-roles/{cr}/grants",
      lambda: pc.grant_privilege(PROBE_CAT, PROBE_CROLE, "TABLE_READ_DATA"), core=True)
probe("mgmt.list_grants", "GET", "/v1/catalogs/{cat}/catalog-roles/{cr}/grants",
      lambda: pc.list_grants(PROBE_CAT, PROBE_CROLE), core=True)
probe("mgmt.list_principals_for_principal_role", "GET",
      "/v1/principal-roles/{r}/principals",
      lambda: pc.list_principals_for_principal_role(PROBE_PROLE))
probe("mgmt.reset_principal_credentials", "POST", "/v1/principals/{p}/reset",
      lambda: pc.reset_principal_credentials(PROBE_PRINCIPAL))
probe("mgmt.delete_catalog_role", "DELETE", "/v1/catalogs/{cat}/catalog-roles/{cr}",
      lambda: pc.delete_catalog_role(PROBE_CAT, PROBE_CROLE))
probe("mgmt.delete_principal_role", "DELETE", "/v1/principal-roles/{r}",
      lambda: pc.delete_principal_role(PROBE_PROLE))
probe("mgmt.delete_principal", "DELETE", "/v1/principals/{p}",
      lambda: pc.delete_principal(PROBE_PRINCIPAL), core=True)

records = tracer.records
print(f"\ntraced {len(records)} operations")

--- Management API: catalogs, principals, roles, grants ---
* mgmt.list_catalogs                           200  sql=16  minio=0   46ms  MISS
* mgmt.get_catalog                             200  sql=14  minio=0   29ms  MISS
* mgmt.create_principal                        201  sql=23  minio=0   35ms  MISS
  mgmt.get_principal                           200  sql=20  minio=0   22ms  MISS
  mgmt.list_principals                         200  sql=29  minio=0   58ms  MISS
  mgmt.create_principal_role                   201  sql=21  minio=0   67ms  MISS
  mgmt.get_principal_role                      200  sql=21  minio=0   32ms  MISS
  mgmt.list_principal_roles                    200  sql=65  minio=0   33ms  MISS
  mgmt.assign_principal_role                   201  sql=24  minio=0   27ms  MISS
  mgmt.create_catalog_role                     201  sql=23  minio=0   35ms  MISS
  mgmt.list_catalog_roles                      200  sql=21  minio=0   17ms  MISS
  mgmt.assign_catalog_role                     20

## 6. Results — API → table and API → MinIO

In [13]:
unknown = unknown_tables(records)
if unknown:
    print(
        f"WARNING: tables outside the expected schema-v2 set were observed: {unknown}\n"
        "Either the SQL parser mis-read a statement, or the deployed schema is not the "
        "version this audit assumes. Resolve before trusting the mapping below."
    )

df_summary = pd.DataFrame(records_to_rows(records))
df_summary["core"] = [getattr(r, "core", False) for r in records]
df_summary.sort_values(["core", "sql_count"], ascending=[False, False]).reset_index(drop=True)

Either the SQL parser mis-read a statement, or the deployed schema is not the version this audit assumes. Resolve before trusting the mapping below.


,api,http_method,path,status,wall_ms,sql_count,sql_total_ms,tables_touched,async_tables,minio_count,minio_total_ms,unaccounted_ms,cache_verdict,error,core
0,iceberg.drop_table,DELETE,/v1/{cat}/namespaces/{ns}/tables/{tbl},204,59.80,87,19.35,"entities,grant_records,nodes,pg_stat_replicati...",,0,NaN,40.45,MISS,None,True
1,mgmt.grant_privilege,PUT,/v1/catalogs/{cat}/catalog-roles/{cr}/grants,201,59.90,48,14.84,"entities,grant_records",,0,NaN,45.06,MISS,None,True
2,iceberg.create_table,POST,/v1/{cat}/namespaces/{ns}/tables,200,70.68,46,8.40,"entities,grant_records",,1,4.13,58.14,MISS,None,True
3,iceberg.drop_namespace,DELETE,/v1/{cat}/namespaces/{ns},204,50.42,33,12.76,"entities,grant_records,policy_mapping_record",,0,NaN,37.66,MISS,None,True
4,mgmt.delete_principal,DELETE,/v1/principals/{p},204,43.10,33,13.50,"entities,grant_records,principal_authenticatio...",,0,NaN,29.60,MISS,None,True
5,iceberg.create_namespace,POST,/v1/{cat}/namespaces,500,54.19,30,10.14,"entities,grant_records",,0,NaN,44.05,MISS,None,True
6,iceberg.commit_table,POST,/v1/{cat}/namespaces/{ns}/tables/{tbl},200,78.08,30,8.26,"entities,grant_records",,3,9.83,59.99,MISS,None,True
7,mgmt.list_grants,GET,/v1/catalogs/{cat}/catalog-roles/{cr}/grants,200,35.16,30,8.24,"entities,grant_records",,0,NaN,26.91,MISS,None,True
8,iceberg.load_table,GET,/v1/{cat}/namespaces/{ns}/tables/{tbl},200,94.29,29,4.63,"entities,grant_records,nodes,pg_stat_replication",,1,1.05,88.60,MISS,None,True
9,iceberg.list_namespaces,GET,/v1/{cat}/namespaces,200,31.86,26,8.00,"entities,grant_records,nodes",,0,NaN,23.86,MISS,None,True


In [14]:
matrix = api_table_matrix(records)
all_tables = sorted({t for row in matrix.values() for t in row})
df_matrix = pd.DataFrame(
    [{"api": api, **{t: matrix[api].get(t, "") for t in all_tables}} for api in matrix]
).set_index("api")
print("R = read, W = write, RW = both\n")
df_matrix

R = read, W = write, RW = both



,entities,grant_records,nodes,pg_stat_replication,policy_mapping_record,principal_authentication_data
api,,,,,,
iceberg.get_config,R,R,,,,
iceberg.list_namespaces,R,R,R,,,
iceberg.load_namespace,R,R,,,,
iceberg.head_namespace,R,R,R,R,,
iceberg.update_namespace_properties,RW,R,,,,
iceberg.create_namespace,RW,R,,,,
iceberg.drop_namespace,RW,RW,,,RW,
iceberg.list_tables,R,R,,,,
iceberg.load_table,R,R,R,R,,


In [15]:
minio_matrix = api_minio_matrix(records)
rows = [
    {"api": api, "method": e["method"], "path": e["path"], "count": e["count"]}
    for api, entries in minio_matrix.items()
    for e in entries
]
if rows:
    display(pd.DataFrame(rows))
else:
    print(
        "No MinIO activity captured. Either the trace stream was not running "
        "(see the preflight warnings), or Polaris genuinely performed no object "
        "I/O for these operations — worth distinguishing before concluding either."
    )

,api,method,path,count
0,iceberg.load_table,,/data-catalog-bucket/apiprofile1787201504_cat/...,1
1,iceberg.load_table[snapshots=refs],,/data-catalog-bucket/apiprofile1787201504_cat/...,1
2,iceberg.head_table,,/data-catalog-bucket/apiprofile1787201504_cat/...,1
3,iceberg.create_table,,/data-catalog-bucket/apiprofile1787201504_cat/...,1
4,iceberg.commit_table,,/data-catalog-bucket/apiprofile1787201504_cat/...,2
5,iceberg.commit_table,,/data-catalog-bucket/apiprofile1787201504_cat/...,1
6,iceberg.create_view,,/data-catalog-bucket/apiprofile1787201504_cat/...,1
7,iceberg.load_view,,/data-catalog-bucket/apiprofile1787201504_cat/...,2
8,iceberg.head_view,,/data-catalog-bucket/apiprofile1787201504_cat/...,2


## 7. Statement inventory and index audit

The distinct statement set, then an `EXPLAIN` verdict for each. Statements are ranked by
`total_ms` (mean × calls), **not** by max duration: the slowest single query is rarely the problem,
whereas a moderately slow query on the authorization path runs on every request.

In [16]:
inventory = statement_inventory(records)
print(f"{len(inventory)} distinct statements\n")
pd.DataFrame(rank_statements(inventory, top=25))[
    ["table", "verb", "calls", "total_ms", "mean_ms", "max_ms", "sql"]
]

23 distinct statements



,table,verb,calls,total_ms,mean_ms,max_ms,sql
0,grant_records,SELECT,59,215.396,3.651,41.056,"SELECT securable_catalog_id, securable_id, gra..."
1,grant_records,DELETE,6,16.990,2.832,4.609,DELETE FROM POLARIS_SCHEMA.GRANT_RECORDS WHERE...
2,entities,SELECT,128,11.186,0.087,1.194,"SELECT id, catalog_id, parent_id, type_code, n..."
3,entities,SELECT,3,7.158,2.386,5.503,"SELECT id, catalog_id, parent_id, type_code, n..."
4,entities,SELECT,111,6.788,0.061,0.485,"SELECT id, catalog_id, parent_id, type_code, n..."
5,entities,SELECT,74,6.347,0.086,0.431,"SELECT id, catalog_id, parent_id, type_code, n..."
6,entities,SELECT,47,3.292,0.070,0.912,"SELECT id, catalog_id, parent_id, type_code, n..."
7,entities,SELECT,1,3.287,3.287,3.287,"SELECT id, catalog_id, parent_id, type_code, n..."
8,entities,INSERT,9,1.166,0.130,0.183,"INSERT INTO POLARIS_SCHEMA.ENTITIES (id, catal..."
9,entities,UPDATE,13,1.151,0.089,0.184,"UPDATE POLARIS_SCHEMA.ENTITIES SET id = ?, cat..."


### Refresh planner statistics before EXPLAIN

`pg_class.reltuples` is only updated by VACUUM/ANALYZE, and this cluster throttles autovacuum
(`autovacuum_naptime = 30s`, `autovacuum_max_workers = 2`). Straight after a seed the planner still
believes the tables are empty, so it picks sequential scans that are correct *for the statistics it
has* and wrong for the data actually present — and the audit would record a false `SEQ_SCAN`.

This is not optional. Skipping it invalidates every index verdict below.

In [17]:
# The module changed on disk; Jupyter caches imports, so a bare re-run of this
# cell would keep using the old analyze_tables. Reload before anything else.
import importlib
import schema_audit
importlib.reload(schema_audit)
from schema_audit import (analyze_tables, statistics_are_stale, table_stats,
                          EXPECTED_TABLES, relation_rowcount)

stale = statistics_are_stale(conn)
if stale:
    print(f"statistics never gathered for: {stale}")
print("running ANALYZE ...")
print("analyzed:", analyze_tables(conn))
if schema_audit.LAST_ANALYZE_ERRORS:
    print("ANALYZE errors:")
    for t, e in schema_audit.LAST_ANALYZE_ERRORS.items():
        print(f"   {t}: {e}")

remaining = statistics_are_stale(conn)

# Scope the gate to the tables the audit actually plans against.
#
# `events` is written by the persistence event listener on a buffered timer and
# is never on any API's synchronous path (api_trace lists it in
# ASYNC_WRITE_TABLES and excludes it from per-API cost). No EXPLAIN below
# touches it, so its statistics cannot make any verdict wrong. Blocking the
# whole notebook on it -- which is what the previous unscoped assert did -- was
# the wrong gate, whatever the underlying reason.
blocking = [t for t in remaining if t in EXPECTED_TABLES]
ignorable = [t for t in remaining if t not in EXPECTED_TABLES]

for t in ignorable:
    try:
        n = relation_rowcount(conn, t)
    except Exception as e:  # noqa: BLE001
        conn.rollback()
        n = f"<count failed: {str(e)[:60]}>"
    with conn.cursor() as cur:
        cur.execute("""SELECT pg_get_userbyid(c.relowner), current_user
                       FROM pg_class c JOIN pg_namespace ns ON ns.oid = c.relnamespace
                       WHERE ns.nspname = 'polaris_schema' AND c.relname = %s""", (t,))
        row = cur.fetchone()
    owner, me = row if row else ("?", "?")
    print(f"  note: '{t}' still unanalyzed (rows={n}, owner={owner}, connected as {me}) "
          f"-- not on any traced path, continuing")
    if owner != me:
        print(f"        ANALYZE needs ownership (or MAINTAIN on PG17); as {me} it is "
              f"skipped with a warning rather than an error, which is why it stays stale.")

assert not blocking, (
    f"no statistics for {blocking} -- these ARE on the traced path, so EXPLAIN "
    "verdicts would be meaningless. Check ownership/permissions before continuing."
)
print("planner statistics are current for every table the audit plans against")


running ANALYZE ...
analyzed: ['entities', 'events', 'grant_records', 'policy_mapping_record', 'principal_authentication_data', 'version']
planner statistics are current for every table the audit plans against


In [18]:
# Attach representative parameters so each statement can actually be EXPLAINed.
#
# First choice: the real bind values, if the trace kept any. It usually did NOT
# -- Polaris logs `?` and api_trace.redact_params strips the values -- which is
# why the previous run returned NO_PARAMS for all 20 statements and produced an
# audit with zero verdicts. An audit that measured nothing reads like a pass.
import importlib
import schema_audit
importlib.reload(schema_audit)
from schema_audit import audit_statements, enrich_inventory
import api_trace
importlib.reload(api_trace)
from api_trace import normalize_sql, statement_inventory, is_polaris_statement

# Rebuild the inventory with the foreign-statement filter applied. log_statement
# ='all' captures repmgr's monitoring, Pgpool's SET/DISCARD handoff and
# pg_stat_replication polling alongside Polaris's own SQL; EXPLAINing those
# produced 19 ERROR rows that buried the verdicts that matter.
inventory = statement_inventory(records)
_dropped = getattr(statement_inventory, "skipped_foreign", 0)
print(f"inventory: {len(inventory)} Polaris statements "
      f"({_dropped} non-Polaris statements filtered out)")

params_by_sql = {}
for rec in records:
    for s in rec.sql:
        key = normalize_sql(s.sql)
        if s.params and key not in params_by_sql:
            params_by_sql[key] = s.params
for entry in inventory:
    if params_by_sql.get(entry["sql"]):
        entry["params"] = params_by_sql[entry["sql"]]

from_trace = sum(1 for e in inventory if e.get("params"))

# Fallback: read the equality columns out of each WHERE clause and take those
# values from a real row of the target table. Statements that cannot be
# resolved honestly (row-constructor IN lists, INSERTs, no WHERE) stay
# NO_PARAMS rather than being EXPLAINed against invented values.
inventory, resolved = enrich_inventory(conn, inventory, realm=REALM)
print(f"parameters: {from_trace} from the trace, "
      f"{resolved - from_trace} sampled from live rows, "
      f"{len(inventory) - resolved} left unresolved (NO_PARAMS)")

audit = audit_statements(conn, inventory)
df_audit = pd.DataFrame(audit)[
    ["verdict", "table", "verb", "calls", "table_rows", "plan_ms",
     "rows_removed_by_filter", "seq_scans", "index_scans", "detail", "sql"]
]
print()
print(df_audit["verdict"].value_counts().to_string())

# Surface any EXPLAIN failure in full -- the previous run showed a bare ERROR
# row for the grant_records DELETE with an empty detail column.
for row in audit:
    if row.get("verdict") == "ERROR":
        print(f"\nERROR on {row.get('table')} {row.get('verb')}:")
        print(f"   {row.get('error') or row.get('detail') or '<no detail captured>'}")
        print(f"   {' '.join(str(row.get('sql', '')).split())[:200]}")

df_audit.sort_values("verdict").reset_index(drop=True)


inventory: 23 Polaris statements (770 non-Polaris statements filtered out)
parameters: 21 from the trace, -3 sampled from live rows, 5 left unresolved (NO_PARAMS)

verdict
INDEX_SCAN    12
NO_PARAMS      5
TOO_SMALL      4
SEQ_SCAN       2


,verdict,table,verb,calls,table_rows,plan_ms,rows_removed_by_filter,seq_scans,index_scans,detail,sql
0,INDEX_SCAN,entities,SELECT,3,7277,1.309,0.0,[],[constraint_name],Index scan via ['constraint_name'].,"SELECT id, catalog_id, parent_id, type_code, n..."
1,INDEX_SCAN,entities,SELECT,111,7277,0.056,0.0,[],[idx_entities],Index scan via ['idx_entities'].,"SELECT id, catalog_id, parent_id, type_code, n..."
2,INDEX_SCAN,entities,SELECT,74,7277,0.038,0.0,[],[constraint_name],Index scan via ['constraint_name'].,"SELECT id, catalog_id, parent_id, type_code, n..."
3,INDEX_SCAN,entities,SELECT,47,7277,0.233,0.0,[],[idx_entities],Index scan via ['idx_entities'].,"SELECT id, catalog_id, parent_id, type_code, n..."
4,INDEX_SCAN,entities,SELECT,1,7277,2.253,0.0,[],[constraint_name],Index scan via ['constraint_name'].,"SELECT id, catalog_id, parent_id, type_code, n..."
5,INDEX_SCAN,entities,INSERT,9,7277,NaN,0.0,[],[],Index scan via [].,"INSERT INTO POLARIS_SCHEMA.ENTITIES (id, catal..."
6,INDEX_SCAN,grant_records,SELECT,24,30013,0.071,0.0,[],[grant_records_pkey],Index scan via ['grant_records_pkey'].,"SELECT securable_catalog_id, securable_id, gra..."
7,INDEX_SCAN,grant_records,INSERT,3,30013,NaN,0.0,[],[],Index scan via [].,INSERT INTO POLARIS_SCHEMA.GRANT_RECORDS (secu...
8,INDEX_SCAN,entities,SELECT,6,7277,0.092,0.0,[],[constraint_name],Index scan via ['constraint_name'].,"SELECT id, catalog_id, parent_id, type_code, n..."
9,INDEX_SCAN,entities,SELECT,3,7277,0.066,0.0,[],[constraint_name],Index scan via ['constraint_name'].,"SELECT id, catalog_id, parent_id, type_code, n..."


In [19]:
# ---------------------------------------------------------------------------
# The IN-list gap: measure the hottest query in the system
# ---------------------------------------------------------------------------
# `SELECT ... FROM entities WHERE (catalog_id, id) IN (...) AND realm_id = ?`
# is the entity cache's bulk validation -- every cached request runs exactly
# one. It reports NO_PARAMS in the table above because normalize_sql collapses
# the id set to a marker that is not valid SQL, so it has never been EXPLAINed.
# Rebuild a real IN-list from live rows and plan it at several lengths: the
# planner can switch strategy as the list grows, so one size proves nothing.
from schema_audit import audit_row_constructor

_in_stmt = next(
    (r for r in audit if "<rows>" in (r.get("sql") or "")
     and r.get("table") == "entities"), None)

if _in_stmt is None:
    print("no collapsed IN-list statement in this capture — nothing to measure")
else:
    print(f"statement ({_in_stmt['calls']} calls):")
    print("  " + " ".join(_in_stmt["sql"].split())[:140])
    print()
    rc = audit_row_constructor(conn, _in_stmt["sql"], realm=REALM,
                               sizes=(1, 10, 50, 200, 500))
    print(f"  {'list':>5} {'verdict':<11} {'node':<24} {'ms':>8}  indexes")
    for r in rc:
        print(f"  {r['n']:>5} {r['verdict']:<11} {str(r.get('node')):<24} "
              f"{(r.get('plan_ms') or 0):>8.3f}  {r.get('indexes') or r.get('error','')}")

    # Feed the result back so check_hypotheses judges this hypothesis on its
    # own statement instead of reporting INCONCLUSIVE for want of a verdict.
    if rc:
        worst = "SEQ_SCAN" if any(r["verdict"] == "SEQ_SCAN" for r in rc) else "INDEX_SCAN"
        for row in audit:
            if row is _in_stmt:
                row["verdict"] = worst
                row["detail"] = ("row-constructor IN measured at sizes "
                                 + ", ".join(str(r["n"]) for r in rc)
                                 + f"; worst plan {worst}")
        print(f"\n  -> recorded {worst} for the hypothesis check")
        if worst == "SEQ_SCAN":
            print("     A sequential scan here taxes EVERY cached request --")
            print("     the cache was supposed to make those cheap.")

    hyp = check_hypotheses(conn, audit)
    print()
    for h in hyp:
        print(f"  {h['id']:32s} {h['status']}")


statement (128 calls):
  SELECT id, catalog_id, parent_id, type_code, name, entity_version, sub_type_code, create_timestamp, drop_timestamp, purge_timestamp, to_purg

   list verdict     node                           ms  indexes
      1 INDEX_SCAN  None                        0.000  
     10 INDEX_SCAN  None                        0.000  
     50 INDEX_SCAN  None                        0.000  
    200 INDEX_SCAN  None                        0.000  
    500 INDEX_SCAN  None                        0.000  

  -> recorded INDEX_SCAN for the hypothesis check

  grant_records_by_grantee         CONFIRMED
  grant_records_delete_or          CONFIRMED
  entities_row_constructor_in      REFUTED


In [20]:
hyps = check_hypotheses(conn, audit)
for h in hyps:
    print(f"[{h['status']:<12}] {h['id']}  (severity: {h['severity']})")
    print(f"    method : {h['source_method']}")
    print(f"    claim  : {h['claim']}")
    print(f"    impact : {h['why_it_matters']}")
    print(f"    remedy : {h['remedy']}")
    for e in h["evidence"][:2]:
        print(f"    evidence: {str(e)[:120]}")
    print()

if any(h["status"] == "INCONCLUSIVE" for h in hyps):
    print(
        "INCONCLUSIVE results usually mean insufficient data volume. Set RUN_SEED = True "
        "in section 3 and re-run — a Seq Scan on a small table is the correct plan and "
        "proves nothing either way."
    )

[CONFIRMED   ] grant_records_by_grantee  (severity: high)
    method : loadAllGrantRecordsOnGrantee
    claim  : grant_records has exactly one index (its PK), which leads with realm_id then the SECURABLE columns. A lookup by GRANTEE cannot use it selectively -- grantee columns sit at positions 4-5 with the securable columns unconstrained in front of them.
    impact : loadAllGrantRecordsOnGrantee runs on the authorization path of EVERY authenticated request, so per-request auth cost would grow with the total number of grants in the realm. Invisible with a handful of grants; steadily worse in a shared realm.
    remedy : CREATE INDEX idx_grant_records_grantee ON grant_records (realm_id, grantee_catalog_id, grantee_id);
    evidence: SELECT securable_catalog_id, securable_id, grantee_catalog_id, grantee_id, privilege_code FROM POLARIS_SCHEMA.GRANT_RECO
    evidence: DELETE FROM POLARIS_SCHEMA.GRANT_RECORDS WHERE ( (grantee_id = ? AND grantee_catalog_id = ?) OR (securable_id = ? AND se

[

In [21]:
stats = table_stats(conn)
print("seq_scan vs idx_scan per table — a high seq_scan_ratio with large seq_tup_read")
print("is the smoking gun for a missing access path, and needs no per-query attribution.\n")
pd.DataFrame(stats)[
    ["table", "n_live_tup", "seq_scan", "seq_tup_read", "idx_scan",
     "seq_scan_ratio", "n_tup_upd", "n_tup_hot_upd", "hot_update_ratio"]
]

seq_scan vs idx_scan per table — a high seq_scan_ratio with large seq_tup_read
is the smoking gun for a missing access path, and needs no per-query attribution.



,table,n_live_tup,seq_scan,seq_tup_read,idx_scan,seq_scan_ratio,n_tup_upd,n_tup_hot_upd,hot_update_ratio
0,grant_records,30013,41586,740669788,20665,0.668,0,0,NaN
1,entities,7277,2795,335589,358576,0.008,65833,65405,0.993
2,principal_authentication_data,1002,234,7768,434,0.350,0,0,NaN
3,version,1,5,5,1,0.833,1,1,1.000
4,policy_mapping_record,0,346,0,14,0.961,0,0,NaN
5,events,1118,0,0,0,NaN,0,0,NaN


## 8. Write the reports

In [22]:
def _write(stem, text):
    """Write <stem>-<stamp>.md, and refresh a stable <stem>-latest.md.

    The timestamped file is the record -- never overwritten, so a before/after
    pair survives. The -latest copy is the convenience handle for whatever
    reads "the current report" without knowing the stamp.
    """
    stamped = REPORTS_DIR / f"{stem}-{RUN_STAMP}.md"
    stamped.write_text(text, encoding="utf-8")
    (REPORTS_DIR / f"{stem}-latest.md").write_text(text, encoding="utf-8")
    print(f"  {stamped}")
    return stamped


def write_reports():
    """Emit doc-api-sql-matrix.md and doc-index-audit.md.

    All content routes through the redacting record objects, never raw log text,
    so no secret material can reach a committed document.
    """
    ts = datetime.now().strftime("%Y-%m-%d %H:%M")

    lines = [
        "# API → SQL → MinIO Access Matrix",
        "",
        f"Generated {ts} from a live local run — Polaris {POLARIS_VERSION}, realm {REALM}.",
        f"Schema version {drift['version_found']} ({drift['verdict']}).",
        "",
        "R = read, W = write, RW = both.",
        "",
        "## API → PostgreSQL tables",
        "",
        "| API | " + " | ".join(all_tables) + " |",
        "|---|" + "---|" * len(all_tables),
    ]
    for api in sorted(matrix):
        lines.append(
            f"| `{api}` | " + " | ".join(matrix[api].get(t, "") or "·" for t in all_tables) + " |"
        )

    lines += ["", "## API → MinIO objects", ""]
    if rows:
        lines += ["| API | Method | Path | Count |", "|---|---|---|---|"]
        lines += [f"| `{r['api']}` | {r['method']} | `{r['path']}` | {r['count']} |" for r in rows]
    else:
        lines.append("_No MinIO activity captured in this run._")

    lines += ["", "## Per-API detail", ""]
    for rec in records:
        lines += [
            f"### `{rec.api}`",
            "",
            f"- `{rec.method} {rec.path}` → **{rec.status}**",
            f"- wall {rec.wall_ms:.0f} ms · {rec.sql_count} statements · "
            f"{rec.minio_count} object ops · cache: {rec.cache_verdict}",
            f"- tables: {', '.join(rec.tables_touched) or '—'}",
            "",
        ]
        if rec.sql:
            lines += ["| # | Table | Verb | ms | SQL |", "|---|---|---|---|---|"]
            for s in rec.sql:
                ms = f"{s.duration_ms:.2f}" if s.duration_ms is not None else "—"
                sql = s.sql.replace("|", "\\|")[:180]
                lines.append(f"| {s.seq} | {s.table or '—'} | {s.verb or '—'} | {ms} | `{sql}` |")
            lines.append("")
        if rec.minio:
            lines += ["| # | Method | Path | ms |", "|---|---|---|---|"]
            for m in rec.minio:
                ms = f"{m.duration_ms:.2f}" if m.duration_ms is not None else "—"
                lines.append(f"| {m.seq} | {m.method} | `{m.path}` | {ms} |")
            lines.append("")

    _write("doc-api-sql-matrix", "\n".join(lines))

    a = [
        "# Index Audit",
        "",
        f"Generated {ts}. Schema version {drift['version_found']}, verdict "
        f"**{drift['verdict']}**.",
        "",
        "## Schema drift",
        "",
    ]
    a += [f"- {n}" for n in drift["notes"]]
    if drift["missing_indexes"]:
        a += ["", "### Missing expected access paths", ""]
        a += [
            f"- `{i['name']}` on `{i['table']}({', '.join(i['columns'])})` [{i['kind']}]"
            for i in drift["missing_indexes"]
        ]

    a += ["", "## Hypotheses", ""]
    for h in hyps:
        a += [
            f"### {h['id']} — **{h['status']}** (severity {h['severity']})",
            "",
            f"- Source: `{h['source_method']}` on `{h['table']}`",
            f"- Claim: {h['claim']}",
            f"- Impact: {h['why_it_matters']}",
            f"- Remedy: `{h['remedy']}`",
            "",
        ]

    a += ["", "## Statement verdicts", "",
          "| Verdict | Table | Verb | Calls | Rows | ms | Detail |", "|---|---|---|---|---|---|---|"]
    for r in sorted(audit, key=lambda x: x["verdict"] or ""):
        ms = f"{r['plan_ms']:.2f}" if r.get("plan_ms") else "—"
        a.append(
            f"| {r['verdict']} | {r['table'] or '—'} | {r['verb'] or '—'} | "
            f"{r['calls']} | {r['table_rows']} | {ms} | {r['detail'][:140]} |"
        )

    _write("doc-index-audit", "\n".join(a))
    print(f"wrote both reports to {REPORTS_DIR}/ with stamp {RUN_STAMP}")


write_reports()

  reports/doc-api-sql-matrix-20260820-135131.md
  reports/doc-index-audit-20260820-135131.md
wrote both reports to reports/ with stamp 20260820-135131


## 9. Teardown and cleanup

Removes the probe fixture, optionally the seeded fixture, and disposes of the database connection
(required by the repo's notebook conventions — the local PgBouncer pool is small).

In [23]:
# Polaris refuses DELETE /catalogs/{name} while the catalog still holds
# namespaces -- it answers 400, not 4xx-with-cascade, and purgeRequested does
# NOT cascade. The captured log shows this plainly: DELETE /mgmt/catalogs/{cat}
# returned 400x3 in doc-api-sql-matrix.md. Empty it from the leaves up.
def drop_catalog_tree(catalog):
    """Delete every table/view/namespace, then the catalog. Returns a report."""
    report = {"tables": 0, "views": 0, "namespaces": 0, "catalog": None}
    r = ic.list_namespaces(catalog)
    if r.status_code == 404:
        report["catalog"] = "absent"
        return report
    for ns in r.json().get("namespaces", []):
        nsn = ns[0] if isinstance(ns, list) else ns
        for t in ic.list_tables(catalog, nsn).json().get("identifiers", []):
            ic.drop_table(catalog, nsn, t["name"], purge=True)
            report["tables"] += 1
        for v in ic.list_views(catalog, nsn).json().get("identifiers", []):
            ic.drop_view(catalog, nsn, v["name"])
            report["views"] += 1
        ic.drop_namespace(catalog, nsn)
        report["namespaces"] += 1
    d = pc.delete_catalog(catalog, purge=True)
    report["catalog"] = d.status_code
    if d.status_code >= 300:
        report["error"] = d.text[:200]
    return report


print("teardown:", drop_catalog_tree(PROBE_CAT))

for name, call in (
    ("principal_role", lambda: pc.delete_principal_role(PROBE_PROLE)),
    ("principal", lambda: pc.delete_principal(PROBE_PRINCIPAL)),
):
    call()

# Earlier runs died at cell 14 before reaching teardown, so their catalogs are
# still there (find_strays reported apiprofile...._cat). Sweep them, since every
# apiprofile* catalog is this notebook's own probe fixture.
leftovers = [c["name"] for c in pc.list_catalogs().json().get("catalogs", [])
             if c["name"].startswith("apiprofile") and c["name"] != PROBE_CAT]
for old in leftovers:
    print(f"  sweeping stale probe catalog {old}: {drop_catalog_tree(old)}")
if not leftovers:
    print("  no stale probe catalogs from earlier runs")

strays = find_strays(pc, prefix=PROBE)
assert not any(strays.values()), f"probe entities left behind: {strays}"

print(f"probe fixture {PROBE_CAT} removed, no residue")


teardown: {'tables': 1, 'views': 0, 'namespaces': 2, 'catalog': 204}
  no stale probe catalogs from earlier runs
probe fixture apiprofile1787201504_cat removed, no residue


In [24]:
TEARDOWN_SEED = False     # <-- set True to delete the ~16k-entity fixture

if TEARDOWN_SEED:
    def on_progress(done, total, res):
        print(f"  torn down {done}/{total}  elapsed={res.elapsed_s:.0f}s")

    tr = teardown(pc, ledger_path=LEDGER, on_progress=on_progress)
    print(tr.summary())
    if tr.failed_users:
        print(f"FAILED (re-run this cell): {tr.failed_users[:5]}")
    else:
        print("seed fixture fully removed; ledger cleared")
elif RUN_SEED:
    print(
        "Seed fixture LEFT IN PLACE — set TEARDOWN_SEED = True when finished.\n"
        f"Ledger: {LEDGER}"
    )

In [25]:
conn.close()
for name in ("df_summary", "df_matrix", "df_audit", "df_idx"):
    if name in dir():
        del globals()[name]
gc.collect()
print("connections closed, frames released")

connections closed, frames released



---

### Reverting the instrumentation

Run the revert section of `doc-instrumentation-runbook.md`. Leaving `log_statement='all'` enabled
will fill the disk and slow every subsequent run.

### Next phases

Deferred by decision (see the plan doc §13): the read-API cache/repeat protocol, the all-CRUD
performance approaches (`/q/metrics` per-API timers, OpenTelemetry span trees, write-path variable
sweeps), and the perspectives beyond DB/SQL/Index. If the grantee-index hypothesis came back
**CONFIRMED**, the follow-up is: create the index locally, re-run this notebook, record the
before/after delta, then check Polaris `main` before filing anything upstream.